### Random forest MTL na reprezentacji fingerprint - zestaw 1 - Absorpcja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. AMES Mutagenicity

Wyniki dla STL:

In [1]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 41.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 97.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pytdc: filename=pytdc-1.1.15-py3-none-any.whl size=191643 sha256=503e0759ae7be324a900d412d5b9b7a94783016da0d8e945f52d9ade226ae28b
  Stored in directory: /root/.cache/pip/wheels/3a/51/0f/0b00fd8b8288143eec76ea0a57804cddd72aae207cc4cb4d65
Successfully built pytdc
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [4]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer): #dodane zabezpieczenia na Nan etc
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                # Zamiast pomijać wiersz, wstawiamy wektor zer (zachowuje wyrównanie)
                fingerprints.append(np.zeros(self.length))

            # Pobieramy etykietę jeśli istnieje (dla dummy X wstawiamy NaN)
            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [10]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [6]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [7]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    # To zapobiega rozbieżnościom rozmiarów X i y
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    # Sprawdzenie czy X zawiera NaN (zabezpieczenie przed błędami featurizera)
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y (z zachowaniem NaN tylko w etykietach!)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        # Tworzymy mapę {Drug: Wynik}
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [9]:
def train_hybrid_mtl_rf(X_train, y_train_dict, X_test, y_test_dict, reg_tasks, class_tasks):
    all_tasks = reg_tasks + class_tasks

    #tworzenie wspólnej macierzy
    Y_train_raw = np.hstack([y_train_dict[task] for task in all_tasks])
    Y_test_raw = np.hstack([y_test_dict[task] for task in all_tasks])

    Y_train_imputed = Y_train_raw.copy()
    Y_test_imputed = Y_test_raw.copy()

    print(">> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...")
    # Aby pozbyć się NaN, trenujemy szybkie modele bazowe.
    #Dla zadań regresyjnych używamy regresora, a dla klasyfikacyjnych klasyfikatora, wyciągając z niego prawdopodobieństwa, aby zachować ciągłość danych.
    for i, task in enumerate(all_tasks):
        known_train_idx = ~np.isnan(Y_train_raw[:, i])
        missing_train_idx = np.isnan(Y_train_raw[:, i])
        missing_test_idx = np.isnan(Y_test_raw[:, i])

        if task in reg_tasks:
            model_single = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i])
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict(X_train[missing_train_idx])
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict(X_test[missing_test_idx])

        elif task in class_tasks:
            model_single = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i].astype(int))
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict_proba(X_train[missing_train_idx])[:, 1]
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict_proba(X_test[missing_test_idx])[:, 1]

    print(">> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...")
    mtl_model = RandomForestRegressor(n_estimators=1000, max_depth=25, max_features="sqrt", random_state=42, n_jobs=-1)
    mtl_model.fit(X_train, Y_train_imputed)

    Y_pred = mtl_model.predict(X_test)

    # Budowanie słownika dopasowanego do funkcji print/save
    metrics_summary = {}
    for i, task in enumerate(all_tasks):
        known_test_idx = ~np.isnan(Y_test_raw[:, i])
        true_y = Y_test_raw[known_test_idx, i]
        pred_y = Y_pred[known_test_idx, i]

        if task in reg_tasks:
            metrics_summary[task] = {
                "task_type": "regression",  # Flaga pomocnicza dla pętli
                "test_metrics": {
                    "rmse": np.sqrt(mean_squared_error(true_y, pred_y)),
                    "mae": mean_absolute_error(true_y, pred_y),
                    "r2": r2_score(true_y, pred_y)
                }
            }
        elif task in class_tasks:
            pred_classes = (pred_y >= 0.5).astype(int)
            true_classes = true_y.astype(int)
            metrics_summary[task] = {
                "task_type": "classification",  # Flaga pomocnicza dla pętli
                "test_metrics": {
                    "accuracy": accuracy_score(true_y, pred_classes),
                    "f1": f1_score(true_y, pred_classes, zero_division=0),
                    "auroc": roc_auc_score(true_y, pred_y)
                }
            }

    return mtl_model, metrics_summary

In [ ]:
# import os

# # 1. Definiowanie konfiguracji i ścieżek (od 2 do 5 endpointów)
# reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca']
# class_tasks = ['HIA_Hou']  # Tutaj dodajesz swoje zadania klasyfikacyjne

# all_tasks = reg_tasks + class_tasks
# filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints_rf.txt"

# # 2. Inicjalizacja Featurizera
# featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# print(">> Ładowanie danych z plików pickle...")
# # Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
# train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
# train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
# train_hia, test_hia = load_split_pickle('HIA_Hou')

# # Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
# df_train_list = [train_caco2, train_lipo, train_hia]
# df_test_list = [test_caco2, test_lipo, test_hia]

# print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
# X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
# X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

# print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
# mtl_model, metrics = train_hybrid_mtl_rf(
#     X_train_mtl, y_train_dict,
#     X_test_mtl, y_test_dict,
#     reg_tasks, class_tasks
# )

# # 3. Zabezpieczenie katalogu docelowego
# dirname = os.path.dirname(filepath)
# if dirname and not os.path.exists(dirname):
#     os.makedirs(dirname)

# # 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
# print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
# for task_name, task_data in metrics.items():
#     task_type = task_data["task_type"]

#     print(f"Endpoint: {task_name} ({task_type.upper()})")

#     # Wywołanie Twojego printu na ekran
#     print_metrics(task_data, task=task_type)

#     # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
#     save_metrics(
#         metrics=task_data,
#         dataset_name=task_name,
#         filepath=filepath,
#         task=task_type,
#         endpoint_group_name="MTL_Absorpcja_Mix"
#     )

# print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

# Test1: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca)

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")

# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_halflife, train_clear]
df_test_list = [test_halflife, test_clear]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Half Life + Clearance Hepatocyte"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

# Test2: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith']  # Dodaj tu zadania klasyfikacyjne
all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_halflife, train_clear, train_cyp]
df_test_list = [test_halflife, test_clear, test_cyp]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Half life + Clearance Hepatocyte + CYP3A4 Inhibition"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

# Test3: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik. Czy w jakimś momencie wyniki przestają się poprawiać?

In [11]:
# --- KONFIGURACJA I URUCHOMIENIE ---
import os
reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/MTL_ML/metryki/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_halflife, train_clear, train_vdss, train_cyp]
df_test_list = [test_halflife, test_clear, test_vdss, test_cyp]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Half life + Clearance Hepatocyte + CYP3A4 Inhibition + Vdss Lombardo"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganGenerator
[20:50:08] DEPRECATION WARNING: please use MorganG

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Half_Life_Obach (REGRESSION)

  RMSE     : 103.2801
  MAE      : 27.4125
  R²       : 0.2564

Endpoint: Clearance_Hepatocyte_AZ (REGRESSION)

  RMSE     : 47.0796
  MAE      : 36.4629
  R²       : 0.1284

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 12.8043
  MAE      : 4.7445
  R²       : -2.4999

Endpoint: CYP3A4_Veith (CLASSIFICATION)

  Accuracy : 0.7174
  F1       : 0.5325
  AUROC    : 0.8243

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/MTL_ML/metryki/mtl_results_eleminacja_fingerprints_rf.txt


# Test4: Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---
import os
reg_tasks = ['Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith']  # Dodaj tu zadania klasyfikacyjne
all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/MTL_ML/metryki/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_clear, train_cyp]
df_test_list = [test_clear, test_cyp]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Clearance Hepatocyte + CYP3A4 Inhibition"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

# Test5: Half Life (Obach) + CYP3A4 Inhibition (Veith)

Sprawdzamy czy duży zbiór Solubility pomoże innemu dużemu zbiorowi mniej niż małemu CaCo-2. Jednocześnie sprawdzamy STL vs MTL.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_halflife, train_cyp]
df_test_list = [test_halflife, test_cyp]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Half Life (Obach) + CYP3A4 Inhibition (Veith)"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

# Test6: Half Life (Obach) + AMES Mutagenicity

Sprawdzamy czy zbiór niepowiązany pomoże zbiorowi CaCo-2.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---
import os
reg_tasks = ['Half_Life_Obach']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/MTL_ML/metryki/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_ames, test_ames = load_split_pickle('AMES')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_halflife, train_ames]
df_test_list = [test_halflife, test_ames]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Half Life + AMES"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganGenerator
[20:10:52] DEPRECATION WARNING: please use MorganG

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Half_Life_Obach (REGRESSION)

  RMSE     : 101.5170
  MAE      : 27.1332
  R²       : 0.2815

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.7524
  F1       : 0.7877
  AUROC    : 0.8459

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/MTL_ML/metryki/mtl_results_eleminacja_fingerprints_rf.txt


# Test7: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo) + AMES Mutagenicity

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith', 'AMES'] # Dodaj tu zadania klasyfikacyjne
all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eleminacja_fingerprints_rf.txt"

# 2. Inicjalizacja Featurizera
featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
# Załaduj dokładnie tyle zestawów, ile zdefiniowałeś w reg_tasks i class_tasks
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')
train_ames, test_ames = load_split_pickle('AMES')

# Zbiory treningowe i testowe w tej samej kolejności co all_tasks!
df_train_list = [train_halflife, train_clear, train_vdss, train_cyp, train_ames]
df_test_list = [test_halflife, test_clear, test_vdss, test_cyp, test_ames]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja, wyświetlenie i zapis przy użyciu Twoich funkcji
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    # Wywołanie Twojego printu na ekran
    print_metrics(task_data, task=task_type)

    # Wywołanie Twojego zapisu (dopisuje wyniki kolejno do pliku tekstowego)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo) + AMES Mutagenicity"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")